# 06 · Standing site monitoring

SAR's killer application is *monitoring*: a satellite re-images the same spot
pass after pass, through cloud and darkness, so the natural workflow is
**standing** — run the same search on a schedule and act only on what is
*new* since last time.

`umbra watch` packages that loop. It is deterministic set arithmetic — **no
model is called** — so it is a clean, testable primitive a cron job, a GitHub
Action, or an agent loop can drive. This notebook wires it into the
standing-analyst pattern the rest of the library was building toward:

> new pass lands → composite it against the previous pass → (optionally)
> narrate the change with a vision model → notify.

It needs the `viz` extra for the change composite:

```bash
pip install "umbra-py[viz]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Pick a site that is imaged repeatedly

An Umbra *task* is repeat imaging of one site, so any task with 2+ passes is a
monitorable target. We sample a modest window and group by task until we find
one — the same pattern `03_change_detection` uses.

In [ ]:
from umbra_py import UmbraCatalog

WINDOW = {"start": "2024-01-01", "end": "2024-12-31"}

by_task: dict[str, list] = {}
for it in UmbraCatalog().search(limit=12, **WINDOW):
    if "GEC" in it.available_assets and it.task:
        by_task.setdefault(it.task, []).append(it)

series = next((v for v in by_task.values() if len(v) >= 2), None)
assert series is not None, "no task with 2+ GEC passes in the sampled window"
site = series[0].task
print(f"monitoring {site!r}: {len(series)} passes in the window")

## 2 · Stand up a watch — the idempotent delta

`watch()` searches an injected source (a live `UmbraCatalog` here), diffs the
results against the set of acquisitions it has already reported, and returns
**only the new ones** — folding them into a state store so the next run knows
what it has seen.

The store is explicit and swappable. `InMemoryWatchStore` is the ephemeral,
offline stand-in we use here; `MetaWatchStore` persists the same state in a
`CatalogIndex`'s `meta` table, so a real standing monitor survives across runs
against the `catalog.db` you already `umbra index fetch`.

`watch_key(...)` derives a stable, readable name from the query, so repeated
runs of the same watch line up on the same stored state.

In [ ]:
from umbra_py import InMemoryWatchStore, watch, watch_key

store = InMemoryWatchStore()
name = watch_key(area=site, product_types=["GEC"])
query = {"area": site, "product_types": ["GEC"], **WINDOW}

first = watch(UmbraCatalog(), name=name, store=store, **query)

# On the first run there is no prior state, so every pass is "new".
assert first.first_run
assert first.new_count == first.total_seen
assert first.new_count >= 2, "need 2+ passes to composite a change"
print(f"first run: {first.new_count} new acquisitions at {site!r}")

## 3 · The standing-analyst guarantee: no re-alerting

Run the exact same watch again with no newly published data and it reports
**zero** new acquisitions. That idempotency is the whole point — a scheduler
can run this every hour and only ever hears about a site when something has
actually landed.

(The delta is an exact set difference over acquisition identities, not a
"latest date seen" watermark, so a late upload dated *earlier* than passes
already reported is still caught — it is genuinely idempotent, not merely
monotonic.)

In [ ]:
again = watch(UmbraCatalog(), name=name, store=store, **query)

assert not again.first_run
assert again.new_count == 0  # nothing new since the run above
assert again.total_seen == first.total_seen
print(f"second run: {again.new_count} new (nothing changed since last check)")

## 4 · Act on the delta — composite the new passes

The new acquisitions from a run are ready to hand to any product verb. Here we
take the standing-analyst action: pick two frames spread across the new passes
and composite the change into one color image.

- **green** = *appeared / brightened* in the later pass,
- **magenta** = *vanished / dimmed*,
- **grey** = unchanged.

In [ ]:
from IPython.display import Image

from umbra_py import save_change_composite, select_change_frames

pair = select_change_frames(first.new_items, frames=2)
assert len(pair) == 2
print("composite:", pair[0].datetime, "->", pair[1].datetime)

png = save_change_composite(pair, "monitor.png", max_size=512)
assert png.exists()
assert png.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"

Image(filename=str(png))

## Where next — schedule it, narrate it

`watch()` supplies the *delta*; the *schedule* is yours to bring:

- **cron / a GitHub Action.** Run `umbra watch --area "<site>" --exit-code`
  on a timer; `--exit-code` turns "are there new acquisitions?" into a process
  exit status a shell `if` can branch on, and `--json` emits
  `{new_count, new_items: [...]}` (with the CC-BY attribution) for the next
  step to act on.
- **Persist across runs.** Swap `InMemoryWatchStore` for `MetaWatchStore(index)`
  so the seen-set lives in your `catalog.db` and a restart does not re-alert.
- **Conversationally.** The same delta is a `watch_site` tool on the `umbra-mcp`
  server, so an agent can ask "what's new at this site?" and feed the answer
  straight into `change_composite` / `describe_scene`.
- **Narrate the change.** `umbra change --narrate` (the `ai` extra) hands this
  composite *and* a deterministic per-block dB grid to a vision model and returns
  a plain-language, number-grounded report of what changed and by how much — so
  the notification carries a picture *and* a sentence, not just a diff.

Together these make SAR-based monitoring available to someone who never has to
learn what a sigma-naught is.

> *Contains Umbra open data, licensed under CC BY 4.0.*